In [10]:
cd /content/drive/MyDrive/HybridRE/Data

/content/drive/MyDrive/HybridRE/Data


In [11]:
import io
from pathlib import Path
from contextlib import redirect_stdout

import pandas as pd
import scorer

# =========================================================
# CONFIG
# =========================================================
BASE_DIR = Path("3_Final_predictions")

PLM = "PLM_ROBERTA-LARGE"
LLM = "Mistral"
SPLIT = "DEV"   # or "DEV"

DATASETS = ["TACRED", "TACREV", "ReTACRED"]
MODES = ["ZEROSHOT", "LORA"]


# =========================================================
# SCORING
# =========================================================
def silent_score(gold, pred):
    f = io.StringIO()
    with redirect_stdout(f):
        p, r, f1 = scorer.score(gold, pred, verbose=False)
    return p, r, f1


def evaluate_file(csv_path):
    df = pd.read_csv(csv_path)

    required_cols = {"True_Labels", "LLM_Prediction"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns {missing} in {csv_path}")

    gold = df["True_Labels"].tolist()
    pred = df["LLM_Prediction"].tolist()

    p, r, f1 = silent_score(gold, pred)
    return p, r, f1


# =========================================================
# FIND FILE
# Expected examples:
# PLM_ROBERTA-LARGE_TACRED_TEST_ZEROSHOT_QWEEN.csv
# PLM_ROBERTA-LARGE_ReTACRED_TEST_LORA_QWEEN.csv
# =========================================================
def find_file(base_dir, dataset, split, mode, llm):
    pattern = f"{PLM}_{dataset}_{split}_{mode}_{llm}.csv"

    # 1) direct recursive exact search
    matches = list(base_dir.rglob(pattern))
    if matches:
        return matches[0]

    # 2) fallback case-insensitive on dataset
    dataset_upper = dataset.upper()
    for path in base_dir.rglob("*.csv"):
        name_upper = path.name.upper()
        target_upper = f"{PLM}_{dataset_upper}_{split}_{mode}_{llm}.CSV".upper()
        if name_upper == target_upper:
            return path

    # 3) more flexible fallback
    for path in base_dir.rglob("*.csv"):
        name_upper = path.stem.upper()
        if (
            name_upper.startswith(f"{PLM}_")
            and f"_{split}_" in name_upper
            and f"_{mode}_" in name_upper
            and name_upper.endswith(f"_{llm}")
        ):
            # extract dataset part between PLM_... and split
            rest = path.stem[len(PLM) + 1:]  # remove "PLM_ROBERTA-LARGE_"
            parts = rest.split("_")
            if len(parts) >= 4:
                ds = "_".join(parts[:-3])   # dataset part
                if ds.upper() == dataset_upper:
                    return path

    return None


# =========================================================
# BUILD FINAL TABLE
# =========================================================
def build_comparison_table(base_dir, datasets, split, llm):
    results = {}

    for dataset in datasets:
        results[dataset] = {}

        for mode in MODES:
            file_path = find_file(base_dir, dataset, split, mode, llm)

            if file_path is None:
                print(f"[WARNING] File not found for: {dataset} | {mode}")
                results[dataset][mode] = {"P": None, "R": None, "F1": None}
                continue

            try:
                p, r, f1 = evaluate_file(file_path)
                results[dataset][mode] = {"P": p, "R": r, "F1": f1}
                print(f"[OK] {dataset} | {mode} -> {file_path.name}")
            except Exception as e:
                print(f"[ERROR] {dataset} | {mode} -> {e}")
                results[dataset][mode] = {"P": None, "R": None, "F1": None}

    # -----------------------------------------------------
    # final rows
    # -----------------------------------------------------
    rows = []

    zero_row = {"Method": f"{llm} ZERO"}
    lora_row = {"Method": f"{llm} LoRA"}
    gain_row = {"Method": "Difference gain (LoRA - Zero)"}

    for dataset in datasets:
        zero_metrics = results[dataset].get("ZEROSHOT", {})
        lora_metrics = results[dataset].get("LORA", {})

        zp, zr, zf = zero_metrics.get("P"), zero_metrics.get("R"), zero_metrics.get("F1")
        lp, lr, lf = lora_metrics.get("P"), lora_metrics.get("R"), lora_metrics.get("F1")

        zero_row[f"{dataset}_P"] = zp
        zero_row[f"{dataset}_R"] = zr
        zero_row[f"{dataset}_F1"] = zf

        lora_row[f"{dataset}_P"] = lp
        lora_row[f"{dataset}_R"] = lr
        lora_row[f"{dataset}_F1"] = lf

        gain_row[f"{dataset}_P"] = None if (lp is None or zp is None) else (lp - zp)
        gain_row[f"{dataset}_R"] = None if (lr is None or zr is None) else (lr - zr)
        gain_row[f"{dataset}_F1"] = None if (lf is None or zf is None) else (lf - zf)

    rows.extend([zero_row, lora_row, gain_row])

    final_df = pd.DataFrame(rows)
    return final_df


# =========================================================
# FORMAT DISPLAY
# =========================================================
def format_as_percent(df):
    out = df.copy()
    for col in out.columns:
        if col != "Method":
            out[col] = out[col].apply(
                lambda x: f"{x:.3%}" if pd.notnull(x) else "N/A"
            )
    return out


# =========================================================
# SAVE
# =========================================================
def save_table(df, base_dir, llm, split):
    out_dir = base_dir / "_summary_tables"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"{PLM}_{llm}_{split}_ZERO_vs_LORA_table.csv"
    df.to_csv(out_path, index=False)

    print(f"\nSaved table to: {out_path}")


# =========================================================
# RUN
# =========================================================
if __name__ == "__main__":
    final_df = build_comparison_table(
        base_dir=BASE_DIR,
        datasets=DATASETS,
        split=SPLIT,
        llm=LLM
    )

    print("\n" + "=" * 120)
    print(f"FINAL COMPARISON TABLE | PLM={PLM} | LLM={LLM} | SPLIT={SPLIT}")
    print("=" * 120)

    display_df = format_as_percent(final_df)
    print(display_df.to_string(index=False))

    save_table(final_df, BASE_DIR, LLM, SPLIT)

[OK] TACRED | ZEROSHOT -> PLM_ROBERTA-LARGE_TACRED_DEV_ZEROSHOT_Mistral.csv
[OK] TACRED | LORA -> PLM_ROBERTA-LARGE_TACRED_DEV_LORA_Mistral.csv
[OK] TACREV | ZEROSHOT -> PLM_ROBERTA-LARGE_TACREV_DEV_ZEROSHOT_Mistral.csv
[OK] TACREV | LORA -> PLM_ROBERTA-LARGE_TACREV_DEV_LORA_Mistral.csv
[OK] ReTACRED | ZEROSHOT -> PLM_ROBERTA-LARGE_RETACRED_DEV_ZEROSHOT_Mistral.csv
[OK] ReTACRED | LORA -> PLM_ROBERTA-LARGE_ReTACRED_DEV_LORA_Mistral.csv

FINAL COMPARISON TABLE | PLM=PLM_ROBERTA-LARGE | LLM=Mistral | SPLIT=DEV
                       Method TACRED_P TACRED_R TACRED_F1 TACREV_P TACREV_R TACREV_F1 ReTACRED_P ReTACRED_R ReTACRED_F1
                 Mistral ZERO  47.179%  60.449%   52.996%  50.266%  66.057%   57.089%    55.040%    57.100%     56.051%
                 Mistral LoRA  75.581%  76.582%   76.078%  83.134%  86.396%   84.734%    92.684%    90.561%     91.611%
Difference gain (LoRA - Zero)  28.402%  16.133%   23.082%  32.868%  20.340%   27.644%    37.644%    33.461%     35.559%

Saved

In [ ]:
import io
from pathlib import Path
from contextlib import redirect_stdout

import pandas as pd
import scorer

# =========================================================
# CONFIG
# =========================================================
BASE_DIR = Path("3_Final_predictions")

PLM = "PLM_ROBERTA-LARGE"
LLM = "Mistral"
SPLIT = "DEV"   # or "DEV"

DATASETS = ["TACRED", "TACREV", "ReTACRED"]
MODES = ["ZEROSHOT", "LORA"]


# =========================================================
# SCORING
# =========================================================
def silent_score(gold, pred):
    f = io.StringIO()
    with redirect_stdout(f):
        p, r, f1 = scorer.score(gold, pred, verbose=False)
    return p, r, f1


def evaluate_file(csv_path):
    df = pd.read_csv(csv_path)

    required_cols = {"True_Labels", "LLM_Prediction"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns {missing} in {csv_path}")

    gold = df["True_Labels"].tolist()
    pred = df["LLM_Prediction"].tolist()

    p, r, f1 = silent_score(gold, pred)
    return p, r, f1


# =========================================================
# FIND FILE
# Expected examples:
# PLM_ROBERTA-LARGE_TACRED_TEST_ZEROSHOT_QWEEN.csv
# PLM_ROBERTA-LARGE_ReTACRED_TEST_LORA_QWEEN.csv
# =========================================================
def find_file(base_dir, dataset, split, mode, llm):
    pattern = f"{PLM}_{dataset}_{split}_{mode}_{llm}.csv"

    # 1) direct recursive exact search
    matches = list(base_dir.rglob(pattern))
    if matches:
        return matches[0]

    # 2) fallback case-insensitive on dataset
    dataset_upper = dataset.upper()
    for path in base_dir.rglob("*.csv"):
        name_upper = path.name.upper()
        target_upper = f"{PLM}_{dataset_upper}_{split}_{mode}_{llm}.CSV".upper()
        if name_upper == target_upper:
            return path

    # 3) more flexible fallback
    for path in base_dir.rglob("*.csv"):
        name_upper = path.stem.upper()
        if (
            name_upper.startswith(f"{PLM}_")
            and f"_{split}_" in name_upper
            and f"_{mode}_" in name_upper
            and name_upper.endswith(f"_{llm}")
        ):
            # extract dataset part between PLM_... and split
            rest = path.stem[len(PLM) + 1:]  # remove "PLM_ROBERTA-LARGE_"
            parts = rest.split("_")
            if len(parts) >= 4:
                ds = "_".join(parts[:-3])   # dataset part
                if ds.upper() == dataset_upper:
                    return path

    return None


# =========================================================
# BUILD FINAL TABLE
# =========================================================
def build_comparison_table(base_dir, datasets, split, llm):
    results = {}

    for dataset in datasets:
        results[dataset] = {}

        for mode in MODES:
            file_path = find_file(base_dir, dataset, split, mode, llm)

            if file_path is None:
                print(f"[WARNING] File not found for: {dataset} | {mode}")
                results[dataset][mode] = {"P": None, "R": None, "F1": None}
                continue

            try:
                p, r, f1 = evaluate_file(file_path)
                results[dataset][mode] = {"P": p, "R": r, "F1": f1}
                print(f"[OK] {dataset} | {mode} -> {file_path.name}")
            except Exception as e:
                print(f"[ERROR] {dataset} | {mode} -> {e}")
                results[dataset][mode] = {"P": None, "R": None, "F1": None}

    # -----------------------------------------------------
    # final rows
    # -----------------------------------------------------
    rows = []

    zero_row = {"Method": f"{llm} ZERO"}
    lora_row = {"Method": f"{llm} LoRA"}
    gain_row = {"Method": "Difference gain (LoRA - Zero)"}

    for dataset in datasets:
        zero_metrics = results[dataset].get("ZEROSHOT", {})
        lora_metrics = results[dataset].get("LORA", {})

        zp, zr, zf = zero_metrics.get("P"), zero_metrics.get("R"), zero_metrics.get("F1")
        lp, lr, lf = lora_metrics.get("P"), lora_metrics.get("R"), lora_metrics.get("F1")

        zero_row[f"{dataset}_P"] = zp
        zero_row[f"{dataset}_R"] = zr
        zero_row[f"{dataset}_F1"] = zf

        lora_row[f"{dataset}_P"] = lp
        lora_row[f"{dataset}_R"] = lr
        lora_row[f"{dataset}_F1"] = lf

        gain_row[f"{dataset}_P"] = None if (lp is None or zp is None) else (lp - zp)
        gain_row[f"{dataset}_R"] = None if (lr is None or zr is None) else (lr - zr)
        gain_row[f"{dataset}_F1"] = None if (lf is None or zf is None) else (lf - zf)

    rows.extend([zero_row, lora_row, gain_row])

    final_df = pd.DataFrame(rows)
    return final_df


# =========================================================
# FORMAT DISPLAY
# =========================================================
def format_as_percent(df):
    out = df.copy()
    for col in out.columns:
        if col != "Method":
            out[col] = out[col].apply(
                lambda x: f"{x:.3%}" if pd.notnull(x) else "N/A"
            )
    return out


# =========================================================
# SAVE
# =========================================================
def save_table(df, base_dir, llm, split):
    out_dir = base_dir / "_summary_tables"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"{PLM}_{llm}_{split}_ZERO_vs_LORA_table.csv"
    df.to_csv(out_path, index=False)

    print(f"\nSaved table to: {out_path}")


# =========================================================
# RUN
# =========================================================
if __name__ == "__main__":
    final_df = build_comparison_table(
        base_dir=BASE_DIR,
        datasets=DATASETS,
        split=SPLIT,
        llm=LLM
    )

    print("\n" + "=" * 120)
    print(f"FINAL COMPARISON TABLE | PLM={PLM} | LLM={LLM} | SPLIT={SPLIT}")
    print("=" * 120)

    display_df = format_as_percent(final_df)
    print(display_df.to_string(index=False))

    save_table(final_df, BASE_DIR, LLM, SPLIT)

[OK] TACRED | ZEROSHOT -> PLM_ROBERTA-LARGE_TACRED_DEV_ZEROSHOT_Mistral.csv
[OK] TACRED | LORA -> PLM_ROBERTA-LARGE_TACRED_DEV_LORA_Mistral.csv
[OK] TACREV | ZEROSHOT -> PLM_ROBERTA-LARGE_TACREV_DEV_ZEROSHOT_Mistral.csv
[OK] TACREV | LORA -> PLM_ROBERTA-LARGE_TACREV_DEV_LORA_Mistral.csv
[OK] ReTACRED | ZEROSHOT -> PLM_ROBERTA-LARGE_RETACRED_DEV_ZEROSHOT_Mistral.csv
[OK] ReTACRED | LORA -> PLM_ROBERTA-LARGE_ReTACRED_DEV_LORA_Mistral.csv

FINAL COMPARISON TABLE | PLM=PLM_ROBERTA-LARGE | LLM=Mistral | SPLIT=DEV
                       Method TACRED_P TACRED_R TACRED_F1 TACREV_P TACREV_R TACREV_F1 ReTACRED_P ReTACRED_R ReTACRED_F1
                 Mistral ZERO  47.179%  60.449%   52.996%  50.266%  66.057%   57.089%    55.040%    57.100%     56.051%
                 Mistral LoRA  75.581%  76.582%   76.078%  83.134%  86.396%   84.734%    92.684%    90.561%     91.611%
Difference gain (LoRA - Zero)  28.402%  16.133%   23.082%  32.868%  20.340%   27.644%    37.644%    33.461%     35.559%

Saved